In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras  # noqa: F401
print(keras.__version__)
print(keras.backend.backend())

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# 直接对原始图像进行归一化
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

# 【关键步】：为卷积层增加“通道轴（Channel Axis）”
# 卷积层要求输入形状为 (样本数, 长, 宽, 通道数)。由于是灰度图，通道数是 1
X_train_conv = X_train_norm[:, :, :, None]
X_test_conv = X_test_norm[:, :, :, None]

print("训练集新形状:", X_train_conv.shape) # 应该显示 (60000, 28, 28, 1)
print("测试集新形状:", X_test_conv.shape) # 应该显示 (10000, 28, 28, 1)

In [ ]:
@keras.saving.register_keras_serializable(package="CustomLayers")
class SwiGLULayer(keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        # 使用 Keras 3 标准的 self.add_weight
        self.W = self.add_weight(
            shape=(input_shape[-1], self.units), 
            initializer="glorot_uniform", 
            trainable=True, 
            name="W"
        )
        self.V = self.add_weight(
            shape=(input_shape[-1], self.units), 
            initializer="glorot_uniform", 
            trainable=True, 
            name="V"
        )
        self.b_W = self.add_weight(
            shape=(self.units,), 
            initializer="zeros", 
            trainable=True, 
            name="b_W"
        )
        self.b_V = self.add_weight(
            shape=(self.units,), 
            initializer="zeros", 
            trainable=True, 
            name="b_V"
        )

    def call(self, inputs):
        # 使用 ops.matmul 代替 tf.matmul
        # 使用 ops.silu 代替 tf.nn.silu
        swish_branch = keras.ops.silu(keras.ops.matmul(inputs, self.W) + self.b_W)
        linear_branch = keras.ops.matmul(inputs, self.V) + self.b_V
        
        # 元素对应相乘（哈达玛积）
        return swish_branch * linear_branch

    def get_config(self):
        # 必须实现 get_config 以支持模型序列化
        config = super().get_config()
        config.update({"units": self.units})
        return config


In [ ]:
# 1. 构建模型（使用 Keras 3 通用层）
model = keras.Sequential([

    keras.layers.Input(shape=(28, 28, 1)), 
    
    keras.layers.Conv2D(16, kernel_size=(3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.SpatialDropout2D(0.25),

    keras.layers.Conv2D(16, kernel_size=(3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.SpatialDropout2D(0.25),
    
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    SwiGLULayer(128),
    keras.layers.GaussianDropout(0.3),
    keras.layers.Dense(10, activation='softmax')

])

In [ ]:
# 模型编译
model.compile(optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# 模型训练并评估
model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test))